In [1]:
import pandas as pd

In [2]:
# load datasets
sales_df = pd.read_parquet('..\\data\\processed\\02_sales_calendar.parquet')
prices = pd.read_csv('..\\data\\raw\\sell_prices.csv')

In [3]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 22 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            str           
 1   item_id       str           
 2   dept_id       str           
 3   cat_id        str           
 4   store_id      str           
 5   state_id      str           
 6   day           str           
 7   sales         uint16        
 8   date          datetime64[us]
 9   wm_yr_wk      uint16        
 10  weekday       str           
 11  wday          uint16        
 12  month         uint16        
 13  year          uint16        
 14  d             str           
 15  event_name_1  str           
 16  event_type_1  str           
 17  event_name_2  str           
 18  event_type_2  str           
 19  snap_CA       uint16        
 20  snap_TX       uint16        
 21  snap_WI       uint16        
dtypes: datetime64[us](1), str(13), uint16(8)
memory usage: 11.5 GB


In [4]:
prices.info()   

<class 'pandas.DataFrame'>
RangeIndex: 6841121 entries, 0 to 6841120
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   store_id    str    
 1   item_id     str    
 2   wm_yr_wk    int64  
 3   sell_price  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 318.1 MB


converting wm_yr_wk from int64 to uint16 in prices dataset to reduce the memory after merge

In [8]:
prices['wm_yr_wk'] = prices['wm_yr_wk'].astype('uint16')

validating there must not be any duplicates on our selected truple condition


In [23]:
prices.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

np.int64(0)

Merging the sales_df and prices on left merge on (store_id,item_id,wm_yr_wk) to make sure that we get correct sell prices from each store each item on the same walmmart week which can later be helpfull in calculating revenue per wallmart week

In [10]:
sales_prices_df = sales_df.merge(prices, how='left', on=['store_id', 'item_id', 'wm_yr_wk'])

In [12]:
sales_prices_df.isnull().sum()

id                     0
item_id                0
dept_id                0
cat_id                 0
store_id               0
state_id               0
day                    0
sales                  0
date                   0
wm_yr_wk               0
weekday                0
wday                   0
month                  0
year                   0
d                      0
event_name_1    53631910
event_type_1    53631910
event_name_2    58205410
event_type_2    58205410
snap_CA                0
snap_TX                0
snap_WI                0
sell_price      12299413
dtype: int64

In [19]:
sales_df['wm_yr_wk'].describe()

count    5.832737e+07
mean     1.133919e+04
std      1.503742e+02
min      1.110100e+04
25%      1.121700e+04
50%      1.133300e+04
75%      1.144800e+04
max      1.161300e+04
Name: wm_yr_wk, dtype: float64

In [20]:
missing_wk_mask = ~sales_df['wm_yr_wk'].isin(prices['wm_yr_wk'])
missing_sales_wk = sales_df[missing_wk_mask]
missing_sales_wk

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,...,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI


there are no missing weeks in the sales data that are not present in the prices data. This means that every week in the sales dataset has a corresponding entry in the prices dataset, ensuring that we can accurately merge the two datasets without losing any sales records due to missing price information.

Because the merge is on three columns, a null `sell_price` means the exact tuple `(store_id, item_id, wm_yr_wk)` was not found in `prices`.
my check only proves `wm_yr_wk` values exist in both datasets, not that every combination of store/item/week exists.

Common reasons:

- `sales_df` has some store/item/week combinations that do not exist in `prices`
- `store_id` or `item_id` values may differ slightly between datasets (spacing, case, formatting)
- `wm_yr_wk` may exist in both, but not for the same store/item pairs
- `prices` may not cover the full range of item/store combinations present in `sales_df`

So the nulls are not because the week is missing globally, but because the full join key is missing for some rows.

In [24]:
sales_prices_df.to_parquet('..\\data\\processed\\03_sales_calendar_prices.parquet', index=False)

In [25]:
sales_prices_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 23 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            str           
 1   item_id       str           
 2   dept_id       str           
 3   cat_id        str           
 4   store_id      str           
 5   state_id      str           
 6   day           str           
 7   sales         uint16        
 8   date          datetime64[us]
 9   wm_yr_wk      uint16        
 10  weekday       str           
 11  wday          uint16        
 12  month         uint16        
 13  year          uint16        
 14  d             str           
 15  event_name_1  str           
 16  event_type_1  str           
 17  event_name_2  str           
 18  event_type_2  str           
 19  snap_CA       uint16        
 20  snap_TX       uint16        
 21  snap_WI       uint16        
 22  sell_price    float64       
dtypes: datetime64[us](1), float64(1), str(13)